In [1]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

model_name = "llama3.1"
chat_generator = OllamaChatGenerator(model=model_name,
                            url = "http://141.2.11.133:11434",
                            timeout = 30*60,
                            generation_kwargs={
                              "temperature": 0,
                              "num_ctx": 8192
                              })

In [4]:
import os
import requests
#  pip install python-dotenv

from typing import List, Dict, Any
from haystack import component
from haystack.dataclasses import Document
from haystack.tools import ComponentTool

from dotenv import load_dotenv
import os

load_dotenv()



@component
class TavilyWebSearch:
    def __init__(self, api_key: str, top_k: int = 3):
        self.api_key = api_key
        self.top_k = top_k

    @component.output_types(documents=List[Document])
    def run(self, query: str) -> Dict[str, Any]:
        """Search Tavily and return Haystack Documents."""
        resp = requests.post(
            "https://api.tavily.com/search",
            json={
                "api_key": self.api_key,
                "query": query,
                "max_results": self.top_k,
                "include_answer": True,
                "include_domains": [
                    "scholar.google.com",
                    "arxiv.org",
                    "researchgate.net",
                ],
                "exclude_domains": ["wikipedia.org"],
            },
            timeout=15,
        )
        resp.raise_for_status()
        data = resp.json()

        docs: List[Document] = []

        if data.get("answer"):
            docs.append(
                Document(
                    content=data["answer"],
                    meta={"source": "tavily:direct_answer"}
                )
            )

        for hit in data.get("results", []):
            docs.append(
                Document(
                    content=hit.get("content", ""),
                    meta={
                        "title": hit.get("title"),
                        "url": hit.get("url"),
                        "source": "tavily:search_result",
                    },
                )
            )

        return {"documents": docs}


web_tool = ComponentTool(
    component=TavilyWebSearch(
        api_key=os.environ["TAVILY_API_KEY"],
        top_k=3,
    ),
    name="web_search",
    description="Live web search via Tavily.",
)

In [9]:
result = agent.run(
    messages=[
        ChatMessage.from_system("You are a helpful research assistant."),
        ChatMessage.from_user("Find papers about the Divergent Association Task.")
    ]
)

#print(result["messages"][-1].content)


In [10]:
for msg in result["messages"]:
    print(msg)

ChatMessage(_role=<ChatRole.SYSTEM: 'system'>, _content=[TextContent(text='\nYou are a research assistant.\nUse web_search whenever you need external information.\nAlways base your answers on retrieved documents.\n')], _name=None, _meta={})
ChatMessage(_role=<ChatRole.SYSTEM: 'system'>, _content=[TextContent(text='You are a helpful research assistant.')], _name=None, _meta={})
ChatMessage(_role=<ChatRole.USER: 'user'>, _content=[TextContent(text='Find papers about the Divergent Association Task.')], _name=None, _meta={})
ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[ToolCall(tool_name='web_search', arguments={'query': 'Divergent Association Task'}, id=None, extra=None)], _name=None, _meta={'model': 'llama3.1', 'done': True, 'total_duration': 1992259500, 'load_duration': 1554388800, 'prompt_eval_duration': 58167500, 'eval_duration': 229382100, 'logprobs': None, 'finish_reason': 'stop', 'completion_start_time': '2026-04-07T12:42:28.3774143Z', 'usage': {'completion_tokens